## Objective & Tasks:
Data Processing: Clean and integrate these datasets. This should include, but not be limited to, handling missing values, duplicates, and possible outliers.

- Load bronze table into a DataFrame.
- Trim all string columns and remove trailing underscores in column names.
- Replace double underscores with single underscores for consistency.
- Convert empty strings to nulls, map boolean columns, and cast all columns to their target types (integer, decimal, date, timestamp, boolean).
- Verify final schema to ensure all columns are correctly typed and ready for further analysis or silver layer storage.
- Check for duplicate entries.

Data source: https://afdc.energy.gov/data_download 

## Outcome:
- A clean, normalized, and type-safe DataFrame ready for further transformations or saving to the silver layer.
- Invalid values are safely set to null to maintain data integrity.
- Column names are standardized

##Step 1: Load the bronze table to further process it in silver layer

In [0]:
df_silver = spark.read.table('adfc_alternative_fuel_stations.bronze')
df_silver.printSchema()

##Stp 2: Trim string columns

Remove leading and trailing spaces from all string columns.

Prevents issues during type conversions (integer, decimal, date, boolean).

In [0]:
from pyspark.sql.functions import trim, col
from pyspark.sql.types import StringType

string_cols = [c.name for c in df_silver.schema.fields if isinstance(c.dataType, StringType)]
for c in string_cols:
    df_silver = df_silver.withColumn(c, trim(col(c)))

##Step 3: Remove trailing underscores from column names

Only removes underscores at the end of column names.

In [0]:
cols_with_trailing_underscore = [c for c in df_silver.columns if c.endswith("_")]
for c in cols_with_trailing_underscore:
    df_silver = df_silver.withColumnRenamed(c, c.rstrip("_"))

##Step 4: Replace double underscores in column names

In [0]:
# Loop through columns and rename if double underscore exists
for c in df_silver.columns:
    if "__" in c:
        new_name = c.replace("__", "_")
        df_silver = df_silver.withColumnRenamed(c, new_name)

df_silver.columns

##Step 5: Convert empty strings to null

- Replaces empty strings in numeric, decimal, date, and timestamp columns with null.
- Ensures that Spark doesn’t fail casting empty strings to numbers/dates.

In [0]:
for c in int_cols + decimal_cols + date_cols + timestamp_cols:
    if c in df_silver.columns:
        df_silver = df_silver.withColumn(c, when(col(c) == "", None).otherwise(col(c)))

##Step 6: Map boolean columns safely
- This code standardizes all boolean-related columns in the DataFrame:
- Loops through each column in bool_cols that exists in the DataFrame.
- Checks if the column type is string or boolean.
- Converts string representations of booleans to proper Boolean values:
   "Y", "y", "true", "True" → True

   "N", "n", "false", "False" → False
- Any other value → null
- Ensures all boolean columns are now consistently typed and ready for analysis.

In [0]:
from pyspark.sql.functions import col, when
from pyspark.sql.types import StringType, BooleanType

for c in bool_cols:
    if c in df_silver.columns:
        dtype = df_silver.schema[c].dataType
        if isinstance(dtype, StringType) or isinstance(dtype, BooleanType):
            df_silver = df_silver.withColumn(
                c,
                when(
                    col(c).cast("string").isin(["Y", "y", "true", "True"]), True
                ).when(
                    col(c).cast("string").isin(["N", "n", "false", "False"]), False
                ).otherwise(None)
            )

df_silver.select(bool_cols).printSchema()


##Step 7: Cast integer columns
- Safely casts numeric columns to IntegerType.
- Invalid values (non-numeric) will become null.

In [0]:
for c in int_cols:
    if c in df_silver.columns:
        df_silver = df_silver.withColumn(c, col(c).cast(IntegerType()))


##Step 8: Cast decimal columns
- Converts latitude and longitude to Decimal(9,6) for precision.
- Ensures numeric calculations are accurate.

In [0]:
for c in decimal_cols:
    if c in df_silver.columns:
        df_silver = df_silver.withColumn(c, col(c).cast(DecimalType(9,6)))


##Step 9: Cast date columns
- Converts string dates to DateType.
- Rows with invalid formats become null.

In [0]:
for c in date_cols:
    if c in df_silver.columns:
        df_silver = df_silver.withColumn(c, to_date(col(c), "yyyy-MM-dd"))

##Step 10: Cast timestamp columns and check final schema
- Converts timestamp strings to TimestampType.
- Ensures proper datetime operations downstream.


## Schema validation
The new schema matches with the documented one from https://afdc.energy.gov/data_download/**historical_stations_format 

- All string columns are trimmed and clean.
- Boolean columns are properly cast (True/False/null).
- Integer columns are correctly typed, with invalid or empty values converted to null.
- Decimal columns (Latitude, Longitude) are properly typed.
- Date and timestamp columns are correctly parsed and typed.
- Column names are clean: no trailing underscores, and double underscores replaced with single underscores where needed.

In [0]:
for c in timestamp_cols:
    if c in df_silver.columns:
        df_silver = df_silver.withColumn(c, col(c).cast(TimestampType()))

df_silver.printSchema()

##Step 11: Checking for nulls and decide how to handle them

- Critical columns → drop rows if null.
- Numeric / counts → fill with 0.
- Categorical / strings → fill with "Unknown" or leave null if meaningful.
- Booleans → fill with False.
- Dates / timestamps → fill with placeholder if needed, else leave null.

In [0]:
from pyspark.sql.functions import col, sum as spark_sum

null_counts = df_silver.select([
    spark_sum(col(c).isNull().cast("int")).alias(c) for c in df_silver.columns
])

display(null_counts)


##Step 12: Type-based null handling

Handle nulls based on column type for logical consistency.

Keep critical data non-null to maintain record integrity.

Use explicit placeholders where missing values may affect computation.

Enable safe, error-free downstream analysis while preserving as much data as possible.

In [0]:
from pyspark.sql.functions import col, lit, sum as spark_sum

# --- 1. Identify column types ---
numeric_cols = [c for c, t in df_silver.dtypes if t in ("int", "bigint", "double", "decimal(9,6)")]
string_cols = [c for c, t in df_silver.dtypes if t == "string"]
bool_cols = [c for c, t in df_silver.dtypes if t == "boolean"]
date_cols = [c for c, t in df_silver.dtypes if t in ("date", "timestamp")]

# --- 2. Critical identifier columns (cannot be null) ---
critical_cols = ["ID", "Station_Name", "Street_Address", "City", "State", "ZIP"]

# --- 3. Fill numeric columns with 0 ---
df_silver = df_silver.fillna({c: 0 for c in numeric_cols})

# --- 4. Fill boolean columns with False ---
df_silver = df_silver.fillna({c: False for c in bool_cols})

# --- 5. Fill non-critical string columns with 'Unknown' ---
string_cols_to_fill = [c for c in string_cols if c not in critical_cols]
df_silver = df_silver.fillna({c: "Unknown" for c in string_cols_to_fill})

# --- 6. Fill date/timestamp columns with a sentinel value (optional) ---
for c in date_cols:
    if dict(df_silver.dtypes)[c] == "date":
        df_silver = df_silver.withColumn(c, col(c).cast("date"))
        df_silver = df_silver.fillna({c: "1970-01-01"})
    else:
        df_silver = df_silver.withColumn(c, col(c).cast("timestamp"))
        df_silver = df_silver.fillna({c: "1970-01-01 00:00:00"})

# --- 7. Drop rows with nulls in critical identifier columns ---
df_silver = df_silver.dropna(subset=critical_cols)

# --- 8. Verify all nulls ---
null_counts = df_silver.select([
    spark_sum(col(c).isNull().cast("int")).alias(c) for c in df_silver.columns
])
display(null_counts)

# --- 9. Confirm schema after cleaning ---
df_silver.printSchema()


## Current State of df_silver

**Non-nullable columns**
Most critical columns (numeric, boolean, many string descriptors, IDs) are now non-nullable, meaning all missing values have been replaced. For example:

- Numeric: EV_Level1_EVSE_Num, CNG_Total_Compression_Capacity → filled with 0.
- Boolean: EV_Workplace_Charging, LPG_Primary → filled with False.
- String (non-critical): EV_Network_Web, Funding_Sources → filled with 'Unknown'.
- Critical IDs & addresses are enforced: ID, Station_Name, Street_Address, City, State, ZIP.

**Nullable columns**
Columns that remain nullable are primarily dates/timestamps and geolocation:

- Latitude, Longitude
- Expected_Date, Date_Last_Confirmed, Open_Date, Data_update_timestamp, Updated_At

These can remain nullable if the original information was missing or optional. Sentinel dates (1970-01-01) may be used to avoid nulls in processing.

**Critical decisions applied**

- Rows missing critical identifiers like ID were dropped, ensuring uniqueness and address integrity.
- Type-based null handling ensures safe downstream processing (aggregations, filtering, joins) without propagation of nulls.

**Ready for analytics**
- Sentinel values in dates allow date calculations without breaking logic.

##Step 13: Column Cleanup & Reordering
**Rename unsafe characters:**

- Replaced - and __ with _ to ensure column names are valid for PySpark operations.
- Avoids errors in expressions, SQL queries, select and filter

**Reorder columns:**

- ID as the first column: makes it easy to identify the primary key.
- Updated_At as the 8th column: places the timestamp in a logical position near other metadata.

**Best practice:**

- Column names are now clean, consistent, and safe.
- The DataFrame is ready for downstream analytics, joins, or exports.

In [0]:
from pyspark.sql.functions import col

# --- 1. Rename columns: replace '-' with '_' and '__' with '_' ---
for c in df_silver.columns:
    new_name = c.replace('-', '_').replace('__', '_')
    if new_name != c:
        df_silver = df_silver.withColumnRenamed(c, new_name)

# --- 2. Reorder columns ---
cols = df_silver.columns

# Ensure 'ID' is first
cols.remove('ID')
cols = ['ID'] + cols

# Ensure 'Updated_At' is 8th
cols.remove('Updated_At')
cols = cols[:7] + ['Updated_At'] + cols[7:]

# Apply reordered columns
df_silver = df_silver.select(cols)

# --- 3. Verify final schema and column order ---
df_silver.printSchema()
display(df_silver)

##Step 14: Checking for duplicate entries based on fied id which is the unique identifier

In [0]:
from pyspark.sql.functions import col, count

# Count occurrences of each ID
duplicate_check = df_silver.groupBy("ID").agg(count("*").alias("count"))

# Filter to see IDs that appear more than once
duplicates = duplicate_check.filter(col("count") > 1)

# Show the duplicates (if any)
duplicates.show(truncate=False)

Found a duplicate ID (0 appears twice). Since ID is supposed to be unique, this is something to address.

In [0]:
df_silver.filter(col("ID") == 0).display(truncate=False)


**Issue:**

Two rows in the dataset have ID = 0, which should be unique.
Critical columns (Station_Name, Street_Address, City, State, ZIP) were checked.

- Row 1: Mostly valid values, assuming ZIP exists.
- Row 2: City is missing/invalid ("""") and ZIP may be missing.

**Decision:**

Since ID = 0 is invalid and at least one row has missing critical information, both rows are dropped.

This ensures:

- All IDs remain unique.
- Critical location and identifier fields are complete.
- Downstream analytics and joins remain safe.

In [0]:
from pyspark.sql.functions import col

# Filter rows with ID = 0 and select critical columns
df_silver.filter(col("ID") == 0).select(critical_cols).display(truncate=False)


In [0]:
from pyspark.sql import functions as F

df = spark.read.table("hive_metastore.adfc_alternative_fuel_stations.silver")

df_gold = df.select([
    F.when(F.col(c).isNull(), "Unknown")
     .when(F.lower(F.col(c)) == "unknown", "Unknown")
     .otherwise(F.col(c))
     .alias(c)
    if t == "string" else F.col(c)
    for c, t in df.dtypes
])

display(df_gold)

**Conclusion:**
- Maintaining unique identifiers is essential for data integrity.
- Dropping invalid or incomplete rows prevents errors in analytics, aggregations, or joins.

In [0]:
from pyspark.sql.functions import col

# Remove rows where ID is 0
df_silver = df_silver.filter(col("ID") != 0)

##Step 15: Write the DataFrame to Bronze Delta Table

In [0]:
# Define the silver path
silver_path = "dbfs:/mnt/afdc/silver"

# Write df_silver to Delta format
df_silver.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .save(silver_path)

print(f"Silver table saved at {silver_path}")

## Step 16: Register the silver table in the metastore for SQL access

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS adfc_alternative_fuel_stations.silver
USING DELTA
LOCATION '{silver_path}'
""")

## Step 17: Display silver table

In [0]:
spark.read.table("hive_metastore.adfc_alternative_fuel_stations.silver").display()